# Image-processing extensions

Port of `apps/demo-ext-image-processing/imgproc.html`. Demonstrates the `@niivue/nv-ext-image-processing` transforms (`otsu`, `connectedLabel`, `conform`, `removeHaze`) running through the widget's bundled extension context.

All four transforms are pre-registered on widget mount, so Python just calls `nv.apply_image_transform(name, ...)`. The transform runs in a Web Worker on the JS side; Python awaits the elapsed-time result.


In [ ]:
import asyncio

import ipywidgets as widgets
from IPython.display import display
from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"

nv = NiiVue(
    background_color=[0.1, 0.1, 0.1, 1.0],
    is_colorbar_visible=True,
    slice_type=3,
    backend="webgl2",
)

# Bundled transforms. Same list is registered on the JS side at mount.
TRANSFORM_NAMES = ["conform", "connectedLabel", "otsu", "removeHaze"]

transform_select = widgets.Dropdown(
    options=[""] + TRANSFORM_NAMES,
    value="",
    description="Transform",
)
options_box = widgets.VBox([])
apply_btn = widgets.Button(description="Apply", disabled=True, button_style="primary")
reset_btn = widgets.Button(description="Reset")
status = widgets.Label(value="Pick a transform.")

current_options_widgets = {}
current_info = None

async def on_transform_change(_change=None):
    global current_info
    name = transform_select.value
    options_box.children = ()
    current_options_widgets.clear()
    current_info = None
    if not name:
        apply_btn.disabled = True
        status.value = "Pick a transform."
        return
    info = await nv.get_volume_transform_info(name)
    current_info = info
    rows = [widgets.HTML(value=f"<i>{(info or {}).get('description', name)}</i>")]
    fields = (info or {}).get("options", []) or []
    for field in fields:
        kind = field.get("type")
        label = field.get("label", field["name"])
        default = field.get("default")
        if kind == "checkbox":
            w = widgets.Checkbox(value=bool(default), description=label)
        elif kind == "select":
            choices = field.get("options") or []
            w = widgets.Dropdown(options=choices, value=default, description=label)
        else:
            w = widgets.Text(value=str(default) if default is not None else "", description=label)
        current_options_widgets[field["name"]] = w
        rows.append(w)
    if not fields:
        rows.append(widgets.Label(value="No configurable options."))
    options_box.children = rows
    apply_btn.disabled = False
    status.value = f"Ready: {name}"

def on_transform_change_sync(_change):
    asyncio.ensure_future(on_transform_change())

async def do_apply():
    name = transform_select.value
    if not name:
        return
    opts = {k: w.value for k, w in current_options_widgets.items()}
    apply_btn.disabled = True
    status.value = f"Running {name}\u2026"
    try:
        # removeHaze replaces the background; segmentations add as overlay.
        result = await nv.apply_image_transform(
            name,
            volume_index=0,
            options=opts,
            replace_background=(name == "removeHaze"),
        )
        ms = result.get("elapsed_ms", 0)
        suffix = " (replaced background)" if name == "removeHaze" else " (added as overlay)"
        status.value = f"{name} done in {ms:.0f} ms{suffix}"
    except Exception as exc:  # noqa: BLE001
        status.value = f"{name} failed: {exc}"
    finally:
        apply_btn.disabled = bool(transform_select.value)

def on_apply(_btn):
    asyncio.ensure_future(do_apply())

def on_reset(_btn):
    nv.remove_all_volumes()
    nv.add_volume_from_url(f"{VOLUMES}/mni152.nii.gz")
    transform_select.value = ""
    status.value = "Reset."

transform_select.observe(on_transform_change_sync, names="value")
apply_btn.on_click(on_apply)
reset_btn.on_click(on_reset)

controls = widgets.VBox([
    widgets.HBox([transform_select, apply_btn, reset_btn]),
    options_box,
])
display(controls)
display(nv)
display(status)

nv.add_volume_from_url(f"{VOLUMES}/mni152.nii.gz")
